<a href="https://colab.research.google.com/github/LokeshPanuganti15/3-2-Training/blob/main/Project_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
!pip install wikipedia sentence-transformers faiss-cpu transformers torch


In [41]:
import wikipedia
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


In [42]:
def fetch_wikipedia(topic):
    return wikipedia.summary(topic, sentences=8)

topic = "Artificial Intelligence"
text = fetch_wikipedia(topic)

print("Wikipedia Text:\n")
print(text)


Wikipedia Text:

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as AI: "A lot of cutting edge AI has filtered into general applications, often wi

In [43]:
def chunk_text(text, chunk_size=80, overlap=20):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunks.append(" ".join(words[i:i + chunk_size]))
    return chunks

chunks = chunk_text(text)

print("Total Chunks:", len(chunks))
print("\nSample Chunk:\n", chunks[0])


Total Chunks: 5

Sample Chunk:
 Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals. High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by


In [44]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embed_model.encode(chunks)
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index built with", index.ntotal, "vectors")


FAISS index built with 5 vectors


In [45]:
def retrieve_chunks(query, k=3):
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]


In [46]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")


In [47]:
def generate_answer(query):
    retrieved_text = retrieve_chunks(query)
    context = " ".join(retrieved_text)

    prompt = f"""
    Context: {context}
    Question: {query}
    Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = gen_model.generate(**inputs, max_length=150)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [48]:
query = "What is Artificial Intelligence?"
answer = generate_answer(query)

print("Question:", query)
print("\nAnswer:", answer)


Question: What is Artificial Intelligence?

Answer: the capability of computational systems to perform tasks typically associated with human intelligence
